# 🚀 Laboratorio di Reinforcement Learning: Missione Lunar Lander
### Progettazione della Funzione di Reward (Reward Engineering)

Benvenuti nel laboratorio di Reinforcement Learning su **Lunar Lander** (Gymnasium).  
In questa attività sperimentale progetterai la formulazione matematica della **funzione di Reward** per guidare un pilota artificiale (agente PPO) ad apprendere in modo autonomo come effettuare un allunaggio morbido e controllato sulla piazzola designata.

---

## 1. L'idea: imparare per tentativi ed errori

Nel **Reinforcement Learning** (apprendimento per rinforzo) l'Intelligenza Artificiale non riceve istruzioni su *cosa* fare, ma impara da sola cercando di massimizzare un punteggio chiamato **Reward**. Ci sono due protagonisti:

| Protagonista | Chi è nel nostro laboratorio |
|---|---|
| **Agente** (chi impara) | Il "pilota" artificiale (algoritmo **PPO**) che controlla i motori |
| **Environment** (il mondo) | Il simulatore fisico *Lunar Lander*: gravità, inerzia, suolo lunare e bandierine |

Il ciclo di controllo si ripete **50 volte al secondo** (ogni *step* dura esattamente $0{,}02\text{ secondi}$):

```text
        ┌───────────────────  osservazione obs (8 sensori fisici)  ───────────────────┐
        ▼                                                                             │
   ┌──────────┐  azione action (0: spento, 1: sx, 2: principale, 3: dx)  ┌─────────────────────────┐
   │  AGENTE  │ ───────────────────────────────────────────────────────▶ │       ENVIRONMENT       │
   │ (pilota) │                                                          │ (modulo + fisica Box2D) │
   └──────────┘ ◀─────────────  REWARD (premio + o penalità -)  ──────── └─────────────────────────┘
```

**Attenzione alla regola d'oro del Reinforcement Learning:**
L'agente non sa cosa significhi "atterrare". Conosce solo i numeri che gli dai tu! Il suo unico obiettivo matematico è **massimizzare la somma totale delle reward ricevute dall'inizio alla fine dell'episodio**:
$$R_{\text{totale}} = r_1 + r_2 + r_3 + \dots + r_T$$
Se scrivi una formula sbilanciata, l'agente troverà delle scorciatoie assurde (*Reward Hacking*) per fare punti senza atterrare!

### 🛠️ Preparazione dell'ambiente locale
Assicurati di aver installato le librerie nel tuo ambiente Python locale:
```bash
pip install "gymnasium[box2d]" stable-baselines3 imageio imageio-ffmpeg tqdm rich
```
Esegui la cella seguente per configurare i percorsi e caricare il motore del laboratorio.


In [ ]:
# Configurazione percorsi per esecuzione in ambiente locale
import os, sys
from pathlib import Path

# Aggiunge la cartella radice del progetto al sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "play2learn_lunarlander" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Imposta la working directory nella root del progetto per sincronizzare cartelle 'runs/'
os.chdir(ROOT_DIR)


In [ ]:
import play2learn_lunarlander
print("✅ Modulo play2learn_lunarlander caricato con successo in ambiente locale!")
print(f"📁 Cartella di lavoro attiva: {ROOT_DIR}")


---
## 2. Conosci la Fisica della Scena e il Sistema di Riferimento

Per progettare una buona funzione matematica devi conoscere le misure esatte dello spazio in cui si muove l'astronave:

* **Origine degli assi $(0.0,\, 0.0)$**: È posizionata **esattamente al centro della piazzola di atterraggio**, a livello del suolo tra le due bandierine.
* **Punto di partenza (Spawn)**: La navicella appare in alto al centro, a circa $(x \approx 0.0,\, y \approx 1.4)$, con una spinta iniziale casuale.
* **Posizione delle Bandierine**: La bandiera sinistra si trova a **`x = -0.20`**, quella destra a **`x = +0.20`**. La zona piatta di atterraggio è quindi larga $0.40$ unità ($|x| \le 0.20$).
* **Confini dello schermo**: I bordi laterali sono a **`x = -1.0`** (sinistra) e **`x = +1.0`** (destra). Se $|x| \ge 1.0$, la navicella esce dallo schermo e l'episodio termina con uno schianto (`is_crashed = True`).
* **A quale velocità si spacca l'astronave?**
  1. **La fusoliera (corpo centrale) è fragilissima**: non deve **mai** toccare il suolo, nemmeno piano, altrimenti esplode subito.
  2. **Le due zampe hanno ammortizzatori**: possono toccare terra, ma se la velocità verticale di caduta è troppo violenta (**`vy < -0.50`**) o la velocità laterale è troppo alta (**`|vx| > 0.25`**) o l'astronave è troppo inclinata (**`|angle| > 0.25` rad** $\approx 14^\circ$), le zampe cedono, la pancia tocca terra e il lander si distrugge! Per un allunaggio morbido ideale, vicino a terra ($y < 0.2$) vogliamo $|v_y| < 0.15$, $|v_x| < 0.10$ e $|\text{angle}| < 0.15$.

Esegui la cella seguente per visualizzare la **Mappa Fisica Annotata** dell'ambiente:

In [ ]:
play2learn_lunarlander.mostra_mappa_ambiente()


---
## 3. Sensori (`obs`), Motori (`action`) ed Eventi di Fine Volo

Ad ogni singolo step ($0{,}02\text{ s}$), la tua funzione `custom_reward(obs, action, is_success, is_crashed, is_timeout)` riceve:

### A. Il vettore `obs` con gli 8 sensori di bordo
| Variabile | Grandezza Fisica | Significato e Valori Tipici | Valore Ideale all'Atterraggio |
| :--- | :--- | :--- | :--- |
| **`x`** | Posizione orizzontale | `-1.0` (bordo sx) $\leftrightarrow$ `0.0` (centro) $\leftrightarrow$ `+1.0` (bordo dx). Bandiere a $\pm 0.20$ | Vicino a `0.0` (tra `-0.2` e `+0.2`) |
| **`y`** | Altitudine (Quota) | `0.0` sulla piazzola al suolo, circa `1.4` alla partenza in alto | `0.0` |
| **`vx`** | Velocità orizzontale | `< 0` va a sinistra, `> 0` va a destra (valori tipici tra `-1.0` e `+1.0`) | `0.0` (fermo lateralmente) |
| **`vy`** | Velocità verticale | `< 0` sta scendendo (caduta), `> 0` sta salendo | Vicino a `0.0` (discesa dolce) |
| **`angle`** | Angolo di assetto | In radianti: `0.0` dritto, `> 0` inclinato a sinistra, `< 0` inclinato a destra | `0.0` (perfettamente verticale) |
| **`vangle`** | Velocità angolare | Velocità di rotazione su se stesso (`< 0` oraria, `> 0` antioraria) | `0.0` (nessuna rotazione) |
| **`left_leg`** | Sensore zampa SX | Vale `1.0` se la zampa sinistra tocca terra, altrimenti `0.0` | `1.0` |
| **`right_leg`** | Sensore zampa DX | Vale `1.0` se la zampa destra tocca terra, altrimenti `0.0` | `1.0` |

### B. Il comando motore `action` scelto dall'agente
| Valore `action` | Azione dei Motori | Effetto Fisico sul Modulo | Consumo Carburante |
| :---: | :--- | :--- | :---: |
| **`0`** | **Motori Spenti** | Il modulo cade per gravità o resta fermo se è già atterrato | `0.0` |
| **`1`** | **Razzo Orientamento SX** | Spinge verso **sinistra** ($v_x$ diminuisce) e inclina a **sinistra** ($\text{angle}$ aumenta) | `0.3` |
| **`2`** | **Motore Principale** | Spinge con forza verso **l'alto** lungo l'asse del modulo per frenare la caduta | `1.0` |
| **`3`** | **Razzo Orientamento DX** | Spinge verso **destra** ($v_x$ aumenta) e inclina a **destra** ($\text{angle}$ diminuisce) | `0.3` |

> 💡 **Perché è fondamentale premiare `action == 0` a terra?**
> Nel simulatore fisico Box2D, l'allunaggio è dichiarato riuscito (`is_success = True`) **solo quando il lander è completamente fermo a riposo**. Se il lander tocca la piazzola con entrambe le zampe ma continua ad accendere i motori (`action != 0`), non andrà mai a riposo e l'episodio finirà per tempo scaduto!

### C. I 3 Flag Booleani di Fine Episodio
* **`is_success`** (`True`/`False`): Allunaggio completato! Il modulo è fermo a riposo sulla piattaforma.
* **`is_crashed`** (`True`/`False`): La fusoliera ha toccato il suolo con violenza oppure il modulo è uscito dai bordi laterali ($|x| \ge 1.0$).
* **`is_timeout`** (`True`/`False`): Sono scaduti tutti i **1000 step** (20 secondi) senza né atterrare né schiantarsi.

### 🔍 Esplorazione: guarda i numeri in tempo reale
Esegui la cella seguente per simulare 10 step con azioni casuali e osservare come variano le 8 coordinate fisiche.

In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("LunarLander-v3")
obs, _ = env.reset(seed=1)

nomi_azioni = {
    0: "0: spento    ",
    1: "1: razzo sx  ",
    2: "2: mot. princ",
    3: "3: razzo dx  "
}

for step in range(10):
    azione = env.action_space.sample()
    obs, reward_std, terminato, scaduto, info = env.step(azione)
    x, y, vx, vy, angle, vangle, left_leg, right_leg = obs

    print(
        f"step {step:2d} | {nomi_azioni[int(azione)]} | "
        f"x={x:6.2f} y={y:6.2f} | vx={vx:6.2f} vy={vy:6.2f} | "
        f"angle={angle:5.2f} rad ({np.degrees(angle):5.1f}°) | "
        f"zampe={int(left_leg)},{int(right_leg)}"
    )

env.close()

---
## 4. La Matematica dei Premi e le 2 Trappole Classiche 🧠

Prima di scrivere la tua funzione, tieni a mente questi **dati di riferimento temporali**:
* 💥 **Caduta libera (Schianto diretto)**: dura circa **60 – 90 step** ($1{,}2 - 1{,}8\text{ s}$).
* 🎯 **Allunaggio controllato ideale**: dura circa **160 – 320 step** ($3 - 6{,}5\text{ s}$).
* ⏳ **Timeout (Tempo massimo)**: scatta esattamente a **1000 step** ($20\text{ s}$).

Poiché la tua funzione `custom_reward` viene chiamata **ad ogni singolo step**, una penalità di `-1.0` ad ogni step ripetuta per 250 step pesa **`-250.0` punti** sul totale! Ecco le **2 trappole matematiche classiche** che devi evitare:

### ⚠️ Trappola 1: Il "Kamikaze" (Suicidio Rapido)
* **L'errore:** Metti penalità ad ogni step troppo alte (es. `reward -= 2.0 * abs(x) - 2.0 * y`) e una penalità di schianto `if is_crashed: reward -= 100.0`.
* **Il calcolo dell'agente:** *"Se provo a frenare e manovrare per 250 step, accumulo $-500$ punti di penalità durante il volo. Se invece mi schianto a tutta velocità in 65 step, accumulo solo $-130$ in volo e $-100$ di schianto = $-230$ punti. Mi conviene buttarmi a capofitto al suolo!"*
* **Come risolverla:** Usa coefficienti piccoli per le penalità ad ogni step (es. tra `0.05` e `0.5`) oppure bilancia bene le penalità temporali con premi per la stabilità e per il contatto delle zampe.

### ⚠️ Trappola 2: L'"Hovering Infinito" (Il fifone che galleggia in aria)
* **L'errore:** Dài un premio positivo a ogni step in volo (es. `reward += 1.0 - abs(x)`), oppure metti `0.0` durante il volo, `-100` per lo schianto e `0` per il timeout.
* **Il calcolo dell'agente:** *"Se resto sospeso a mezz'aria sopra le bandiere fino a far scadere i 1000 step, incasso punti ad ogni istante senza mai correre il rischio di toccare terra e prendere -100!"*
* **Come risolverla:** Durante il volo la reward per step dovrebbe essere **leggermente negativa** (incentivando l'agente a scendere verso `y = 0` e concludere la missione), e il timeout (`is_timeout`) va punito adeguatamente!

---
## 5. Percorso Guidato a Livelli: Progetta la tua `custom_reward`

Ti proponiamo di costruire la tua funzione per gradi:
* **🥈 Livello 1 (Allineamento e Discesa):** Gestisci gli eventi finali (`is_success`, `is_crashed`, `is_timeout`) e aggiungi piccole penalità ad ogni step per la distanza dal centro `abs(x)`, l'altezza `y` e l'inclinazione `abs(angle)`.
* **🥇 Livello 2 (Controllo Velocità):** Aggiungi il controllo sulla velocità orizzontale `abs(vx)`, verticale `abs(vy)` e di rotazione `abs(vangle)`.
* **💎 Livello 3 (Atterraggio Morbido):** Premia il contatto delle zampe (`left_leg`, `right_leg`) quando la discesa è lenta e penalizza lo spreco eccessivo di carburante.


In [ ]:
def custom_reward(obs, action, is_success, is_crashed, is_timeout):
    """
    Progetta qui la tua funzione matematica di Reward!

    SENSORI FISICI (obs):
      x, y           -> Posizione orizzontale (-1..+1, bandiere a ±0.2) e quota (0=piazzola, ~1.4=cima)
      vx, vy         -> Velocità orizzontale e verticale (vy < 0 sta scendendo)
      angle, vangle  -> Angolo rispetto alla verticale (rad) e velocità di rotazione
      left_leg, right_leg -> Sensori di contatto zampe (1.0 = tocca terra, 0.0 = in aria)

    AZIONE SCELTA (action):
      0 = Motori spenti | 1 = Razzo SX | 2 = Motore Principale | 3 = Razzo DX

    EVENTI DI FINE EPISODIO (True / False):
      is_success, is_crashed, is_timeout
    """
    x, y, vx, vy, angle, vangle, left_leg, right_leg = obs
    reward = 0.0

    # =========================================================================
    # SCRIVI QUI LA TUA FORMULA DI REWARD SHAPING (Modifica e sperimenta!)
    # =========================================================================





    # Grande premio se l'allunaggio riesce!
    if is_success:
        reward += 100.0

    # Forte penalità se il modulo si distrugge!
    if is_crashed:
        reward -= 100.0

    return float(reward)

print("✅ Funzione custom_reward salvata! Passa alla cella successiva per avviare l'addestramento.")


---
## 6. Addestramento e Telemetria di Volo

Durante l'addestramento vedrai la tabella di *rollout* con tre indicatori chiave:
* **`rollout/ep_rew_mean`**: La media della **tua custom reward** negli ultimi episodi.
* **`rollout/ep_len_mean`**: Quanti step dura in media un volo (ricorda: `< 90` = kamikaze, `160-320` = ottimo, `> 850` = hovering infinito).
* **`rollout/success_rate`**: La frazione di allunaggi riusciti (`0.0` = 0%, `1.0` = 100%).

Al termine verranno mostrati la **Dashboard con 4 grafici**, la **Diagnosi del Flight Coach** e i **video di prova**!

In [ ]:
# Avvia l'addestramento PPO con la tua custom_reward
# Hint: impostiamo 300.000 step di base (i primi atterraggi compaiono intorno ai 200k step, mentre verso i 300k si consolida un comportamento emergente stabile e pulito con successo >= 80%).
# Se il tasso di successo finale è >= 80%, verrà generato automaticamente anche un report completo alla fine della run.
risultati = play2learn_lunarlander.run_lab(
    reward_fn=custom_reward,
    mission="luna_standard",
    total_timesteps=300_000,
    nome_run="addestramento_base"
)

---
## 7. 🏆 Livelli Sfida Avanzati: Metti alla prova il tuo Pilota!

Sei riuscito a superare il livello base con almeno l'**80% di allunaggi riusciti**? Ora scegli una delle **Missioni Speciali** modificando la tua `custom_reward` e cambiando il parametro `mission` nella cella qui sotto:

1. ⛽ **`mission="eco_lander"` (Risparmio Carburante)**: Atterra con successo consumando **meno di 60 unità di carburante** in media! *(Hint: penalizza `action == 2` con un piccolo costo ad ogni accensione del motore principale e `action in [1, 3]` per i razzi laterali)*.
2. ⚡ **`mission="speedrun"` (Atterraggio Rapido)**: Atterra sano e salvo in **meno di 160 step** (3,2 secondi)! *(Hint: fai scendere il lander più velocemente quando è ad alta quota `y > 0.5`, ma costringilo a frenare bruscamente quando `y < 0.4`)*.
3. 🌪️ **`mission="tempesta_solare"` (Vento Forte + Turbolenza)**: Raffiche di vento laterale spingono la navicella fuori rotta e la fanno ruotare. *(Hint: dovrai rendere più severo il controllo sull'assetto `angle`, sulla velocità angolare `vangle` e sulla deriva `vx`)*.
4. 🪐 **`mission="pianeta_pesante"` (Super-Gravità `-11.8`)**: La gravità tira giù il modulo con molta più forza.
5. 🔴 **`mission="marte_leggero"` (Micro-Gravità `-4.5` + Turbolenza)**: Gravità debole: se usi troppo il motore principale (`action == 2`), la navicella rimbalza verso l'alto (`vy > 0`) e vola via!

In [ ]:
# Scegli la missione tra:
# "eco_lander", "speedrun", "tempesta_solare", "pianeta_pesante", "marte_leggero"

risultati_sfida = play2learn_lunarlander.run_lab(
    reward_fn=custom_reward,
    mission="tempesta_solare",
    total_timesteps=300_000,
    nome_run="sfida_tempesta"
)


---
## 8. 📊 Reportistica, Debug e Confronto tra Esperimenti

Quando sperimenti con la funzione di reward, è utile poter salvare le run migliori, generare una scheda riassuntiva dei dati e confrontare graficamente i diversi tentativi.

In questa sezione puoi:
* 💾 **Salvare i modelli preferiti** con le tue annotazioni personali.
* 📑 **Generare una Scheda Tecnica (PDF, HTML, Markdown)**: include la telemetria dettagliata, il codice della reward e uno spazio per le tue considerazioni (i file generati sono modificabili a piacere!).
* 📈 **Confrontare due o più run**: visualizza a colpo d'occhio se le modifiche hanno aumentato il successo o ridotto i consumi.


In [ ]:
# 1. Salva la run completata nell'archivio dei modelli preferiti
risultati.salva_preferito(
    nome_personalizzato="Allunaggio_Esperimento_1",
    note="Prima versione stabile: controllo discesa e quota"
)


In [ ]:
# 2. Visualizza la tabella riassuntiva di tutti i modelli salvati
play2learn_lunarlander.mostra_preferiti()


In [ ]:
# 3. Genera la Scheda Tecnica dell'esperimento (PDF, HTML e Markdown)
# I file vengono salvati in una cartella dedicata per poter essere consultati e modificati
report = risultati.genera_report(
    autore="Nome Studente / Gruppo",
    titolo="Esperimento Allunaggio Lunar Lander",
    note="Inserisci qui le tue osservazioni sul comportamento dell'agente e sulle modifiche alla reward"
)


In [ ]:
# 4. Metti a confronto due o più run per analizzare l'impatto delle modifiche matematiche
play2learn_lunarlander.confronta_run([risultati, risultati_sfida])
